In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

**Read Holiday landing data**

In [ ]:
holiday_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/bronze/holiday/"

display(dbutils.fs.ls(holiday_path))

In [ ]:
df_holiday_raw = (
    spark.read
    .option("multiline", "true")
    .format('delta').load(holiday_path)
)

print("Column count:", len(df_holiday_raw.columns))
print("Columns:", df_holiday_raw.columns)

display(df_holiday_raw)

**Holiday landing**

In [ ]:
from pyspark.sql.functions import col, to_date, concat_ws, coalesce, lit

df_holiday_bronze = (
    df_holiday_raw
    .select(
        col("date").alias("holiday_date"),
        col("countryCode").alias("country_code"),
        col("localName").alias("local_name"),
        col("name").alias("holiday_name"),
        col("fixed").cast("boolean").alias("is_fixed"),
        col("global").cast("boolean").alias("is_global"),
        col("launchYear").cast("int").alias("launch_year"),
        col("counties").alias("counties"),
        col("types").alias("holiday_types")
    )
)

display(df_holiday_bronze)

## # **Write landing**

In [ ]:
holiday_bronze_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "bronze/holiday/"
)

"""(
    df_holiday_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(holiday_bronze_path)
)

print("Holiday landing written successfully")
display(
    spark.read.format("delta").load(holiday_bronze_path)
)"""

## ** Holiday Silver**

In [ ]:
# Re-read bronze after schema overwrite in Cell 8
df_holiday_bronze = spark.read.format("delta").load(holiday_bronze_path)

df_holiday_silver = (
    df_holiday_bronze
    .withColumn("holiday_date", to_date(col("holiday_date")))
    .withColumn(
        "county_scope",
        coalesce(
            concat_ws(",", col("counties")),
            lit("ALL")
        )
    )
    .dropDuplicates([
        "holiday_date",
        "country_code",
        "local_name",
        "holiday_name",
        "county_scope"
    ])
)

display(df_holiday_silver)

## Holiday DQ checks

In [ ]:
print("Holiday Silver row count:", df_holiday_silver.count())

"""Null holiday dates"""
print(
    "Null holiday dates:",
    df_holiday_silver.filter(col("holiday_date").isNull()).count()
)

In [ ]:
"""Invalid country codes"""
print(
    "Invalid country codes:",
    df_holiday_silver
    .filter(col("country_code").isNull() | (col("country_code") != "US"))
    .count()
)

In [ ]:
"""Duplicate Holiday record"""
duplicate_count = (
    df_holiday_silver
    .groupBy(
        "holiday_date",
        "country_code",
        "local_name",
        "holiday_name",
        "county_scope"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Holiday records:", duplicate_count)

In [ ]:
print(
    "Invalid fixed/global values:",
    df_holiday_silver.filter(
        col("is_fixed").isNull() |
        col("is_global").isNull()
    ).count()
)

## Write Holiday Silver

In [ ]:
holiday_silver_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "silver/holiday/"
)

(
    df_holiday_silver
    .write
    .format("delta")
    .mode("overwrite")
    .save(holiday_silver_path)
)

print("Holiday Silver written successfully")

In [ ]:
"""Verify"""
df_holiday_silver_check = (
    spark.read
    .format("delta")
    .load(holiday_silver_path)
)

print("Silver row count:", df_holiday_silver_check.count())
display(df_holiday_silver_check.orderBy("holiday_date"))